# RCT Outcome Prediction — EDA
**데이터**: `/Volumes/k7kbot1_SSD/CVD_OUTCOME_sec`  
**파일 구조**: `{pmid}_pri.json` (primary outcome) + `{pmid}_sec.json` (secondary outcomes)

In [1]:
import json, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
from pathlib import Path

DATA_DIR = Path('/Volumes/k7kbot1_SSD/CVD_OUTCOME_sec')
sns.set_theme(style='whitegrid')
print('Libraries loaded')

Matplotlib is building the font cache; this may take a moment.


Libraries loaded


## 1. 데이터 로딩

In [ ]:
pri_files = sorted(DATA_DIR.glob('*_pri.json'))
sec_files = sorted(DATA_DIR.glob('*_sec.json'))
print(f'_pri 파일 수: {len(pri_files)}')
print(f'_sec 파일 수: {len(sec_files)}')

def load_json_files(file_list, label):
    records = []
    for f in file_list:
        with open(f) as fp:
            d = json.load(fp)
            d['_source'] = label
            records.append(d)
    return records

pri_records = load_json_files(pri_files, 'primary')
sec_records = load_json_files(sec_files, 'secondary')

df_pri = pd.DataFrame(pri_records)
df_sec = pd.DataFrame(sec_records)

print(f'\ndf_pri shape: {df_pri.shape}')
print(f'df_sec shape: {df_sec.shape}')
print(f'\ndf_pri columns: {list(df_pri.columns)}')
print(f'df_sec columns: {list(df_sec.columns)}')

In [ ]:
# _status 확인
print('_pri _status:', df_pri['_status'].value_counts().to_dict())
print('_sec _status:', df_sec['_status'].value_counts().to_dict())

# 유효 데이터만
df_pri = df_pri[df_pri['_status'] == 'success'].copy()
df_sec = df_sec[df_sec['_status'] == 'success'].copy()
print(f'\n유효 pri: {len(df_pri):,}  /  유효 sec: {len(df_sec):,}')

## 2. 파싱 함수 정의

In [ ]:
def detect_metric(text, metric_hint=''):
    """HR / RR / OR / MD / SMD / RD / OTHER 판별"""
    combined = (metric_hint + ' ' + text).upper()
    if any(k in combined for k in ['HAZARD RATIO', ' HR ', 'HR=', 'HR:']):
        return 'HR'
    if any(k in combined for k in ['RELATIVE RISK', ' RR ', 'RR=', 'RR:']):
        return 'RR'
    if any(k in combined for k in ['ODDS RATIO', ' OR ', 'OR=', 'OR:']):
        return 'OR'
    if any(k in combined for k in ['STANDARDIZED MEAN', ' SMD ', 'COHEN']):
        return 'SMD'
    if any(k in combined for k in ['MEAN DIFFERENCE', ' MD ', 'LEAST SQUARES', 'DIFFERENCE IN MEAN']):
        return 'MD'
    if any(k in combined for k in ['RISK DIFFERENCE', ' RD ']):
        return 'RD'
    return 'OTHER'


def extract_first_number(text):
    """문자열에서 첫 번째 유효 숫자 추출"""
    nums = re.findall(r'-?\d+\.\d+|-?\d+', text)
    return float(nums[0]) if nums else None


def extract_direction(text):
    """Favors ... 추출"""
    if 'No significant difference' in text or 'No difference' in text:
        return 'No difference'
    m = re.search(r'Favors\s+([\w\s\-]+?)(?:\)|;|,|$)', text)
    return m.group(1).strip() if m else None


def parse_ci(ci_str):
    """'0.61-0.88' or '0.61–0.88' → (0.61, 0.88)"""
    if not isinstance(ci_str, str):
        return None, None
    nums = re.findall(r'-?\d+\.\d+|-?\d+', ci_str)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    return None, None


def parse_pvalue(p_str):
    if not isinstance(p_str, str):
        return None
    # 'p < 0.001' 처럼 부등호 포함 케이스
    nums = re.findall(r'\d+\.\d+(?:e[+-]?\d+)?|\d+', p_str.lower())
    try:
        return float(nums[0]) if nums else None
    except:
        return None


def parse_pri_result(row):
    """_pri 레코드 → 단일 결과 dict"""
    val_str = str(row.get('primary_result_value', '') or '')
    metric_hint = str(row.get('primary_result_metric', '') or '')
    metric = detect_metric(val_str, metric_hint)
    value  = extract_first_number(val_str)
    direction = extract_direction(val_str)
    ci_l, ci_u = parse_ci(row.get('primary_ci_95'))
    pval = parse_pvalue(row.get('primary_p_value'))

    if value is None:
        return None
    if metric in ('HR', 'RR', 'OR') and not (0 < value < 20):
        return None

    return {
        'pmid': row['pmid'],
        'source': 'primary',
        'metric': metric,
        'value': value,
        'ci_lower': ci_l,
        'ci_upper': ci_u,
        'p_value': pval,
        'direction': direction,
        'outcome_text': str(row.get('primary_outcome', '') or ''),
    }


def parse_sec_results(row):
    """
    _sec 레코드 → 복수 결과 list
    primary_result_value가 세미콜론으로 구분된 복수 결과
    """
    val_str = str(row.get('primary_result_value', '') or '')
    ci_str  = str(row.get('primary_ci_95', '') or '')
    p_str   = str(row.get('primary_p_value', '') or '')
    outcomes = row.get('primary_outcome', [])
    if isinstance(outcomes, str):
        outcomes = [outcomes]

    # 세미콜론으로 분리
    parts    = [s.strip() for s in val_str.split(';') if s.strip()]
    ci_parts = [s.strip() for s in ci_str.split(';')  if s.strip()]
    p_parts  = [s.strip() for s in p_str.split(';')   if s.strip()]

    results = []
    for i, part in enumerate(parts):
        metric    = detect_metric(part)
        value     = extract_first_number(part)
        direction = extract_direction(part)
        ci_l, ci_u = parse_ci(ci_parts[i] if i < len(ci_parts) else None)
        pval = parse_pvalue(p_parts[i] if i < len(p_parts) else None)
        outcome_text = outcomes[i] if i < len(outcomes) else ''

        if value is None:
            continue
        if metric in ('HR', 'RR', 'OR') and not (0 < value < 20):
            continue

        results.append({
            'pmid': row['pmid'],
            'source': 'secondary',
            'metric': metric,
            'value': value,
            'ci_lower': ci_l,
            'ci_upper': ci_u,
            'p_value': pval,
            'direction': direction,
            'outcome_text': str(outcome_text),
        })
    return results


print('파싱 함수 정의 완료')

## 3. 레이블 파싱 — HR / RR / OR / MD 추출

In [ ]:
# Primary 파싱
pri_results = [parse_pri_result(row) for _, row in df_pri.iterrows()]
pri_results = [r for r in pri_results if r]

# Secondary 파싱
sec_results = []
for _, row in df_sec.iterrows():
    sec_results.extend(parse_sec_results(row))

df_label = pd.DataFrame(pri_results + sec_results)

print(f'Primary 파싱 성공:   {len(pri_results):,} / {len(df_pri):,}')
print(f'Secondary 파싱 성공: {len(sec_results):,} / {len(df_sec):,}')
print(f'\n전체 레이블 수: {len(df_label):,}')
print(f'\n지표별 분포:')
print(df_label['metric'].value_counts())

In [ ]:
# 지표 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Primary vs Secondary 비교
metric_pri = df_label[df_label['source']=='primary']['metric'].value_counts()
metric_sec = df_label[df_label['source']=='secondary']['metric'].value_counts()
compare = pd.DataFrame({'Primary': metric_pri, 'Secondary': metric_sec}).fillna(0)
compare.plot(kind='bar', ax=axes[0], color=['#2196F3','#FF9800'], edgecolor='white')
axes[0].set_title('지표별 분포 (Primary vs Secondary)', fontsize=12)
axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# HR/RR/OR 값 분포
ratio_df = df_label[df_label['metric'].isin(['HR','RR','OR'])]
ratio_df['value'].clip(0.1, 5).hist(bins=80, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='No effect (=1)')
axes[1].set_title(f'HR/RR/OR 분포 (n={len(ratio_df):,})', fontsize=12)
axes[1].set_xlabel('Effect Size (0.1–5 clip)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_metric_distribution.png', dpi=150)
plt.show()

## 4. 모델 학습용 Clean Subset

In [ ]:
# HR/RR/OR + CI + p-value 모두 있는 subset
model_df = df_label[
    df_label['metric'].isin(['HR','RR','OR']) &
    df_label['value'].notna() &
    df_label['ci_lower'].notna() &
    df_label['ci_upper'].notna() &
    df_label['p_value'].notna()
].copy()

model_df['log_effect'] = np.log(model_df['value'])
model_df['significant'] = (model_df['p_value'] < 0.05).astype(int)

print('=== 모델 학습 가능 데이터 요약 ===')
print(f'총 샘플        : {len(model_df):,}')
print(f'Primary 출처   : {(model_df["source"]=="primary").sum():,}')
print(f'Secondary 출처 : {(model_df["source"]=="secondary").sum():,}')
print(f'\n지표별:')
print(model_df['metric'].value_counts())
print(f'\n유의한 결과 (p<0.05) : {model_df["significant"].sum():,} ({model_df["significant"].mean()*100:.1f}%)')
print(f'log(effect) 평균      : {model_df["log_effect"].mean():.3f}')
print(f'log(effect) 표준편차  : {model_df["log_effect"].std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# log(effect) 분포
model_df['log_effect'].hist(bins=80, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='log(1)=No effect')
axes[0].set_title('log(HR/RR/OR) 분포', fontsize=12)
axes[0].set_xlabel('log(effect size)')
axes[0].legend()

# 유의성 분포
sig_counts = model_df['significant'].value_counts().reindex([0,1], fill_value=0)
sig_counts.plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_xticklabels(['Not sig. (p≥0.05)', 'Sig. (p<0.05)'], rotation=0)
axes[1].set_title('유의성 분포', fontsize=12)
axes[1].set_ylabel('Count')

# Primary vs Secondary 비율
model_df['source'].value_counts().plot(
    kind='pie', ax=axes[2], autopct='%1.1f%%',
    colors=['#2196F3','#FF9800'], startangle=90
)
axes[2].set_title('Primary vs Secondary', fontsize=12)
axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/eda_label_distribution.png', dpi=150)
plt.show()

## 5. 환자 특성 Feature 분석

In [ ]:
def parse_patient_features(feature_list):
    result = {}
    if not isinstance(feature_list, list):
        return result
    for item in feature_list:
        mesh_match = re.search(r'\(MeSH:\s*([^)]+)\)', item)
        mesh = mesh_match.group(1).strip() if mesh_match else None
        name_part = re.sub(r'\(MeSH:[^)]+\)', '', item)
        name_match = re.match(r'\[([^:]+):', name_part)
        name = name_match.group(1).strip() if name_match else None
        nums = re.findall(r'-?\d+\.\d+|-?\d+', item)
        value = float(nums[0]) if nums else None
        unit_match = re.search(r'\(([^()]+)\)\s*\]', item)
        unit = unit_match.group(1) if unit_match else None
        if unit and 'MeSH' in unit:
            unit = None
        key = mesh if mesh else name
        if key and value is not None:
            result[key] = {'name': name, 'value': value, 'unit': unit}
    return result

# pri 데이터로 feature 빈도 집계
feature_counter = Counter()
for _, row in df_pri.iterrows():
    feats = parse_patient_features(row.get('patient_treatment'))
    feature_counter.update(feats.keys())

print(f'고유 feature 수: {len(feature_counter):,}')
print(f'\n상위 30개 feature (빈도 / 커버리지):')
for feat, cnt in feature_counter.most_common(30):
    print(f'  {cnt:5d} ({cnt/len(df_pri)*100:5.1f}%)  {feat}')

In [ ]:
top_n = 30
top_features = feature_counter.most_common(top_n)
feat_names  = [f[0][:45] for f in top_features]
feat_counts = [f[1] for f in top_features]

fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(feat_names[::-1], feat_counts[::-1], color=sns.color_palette('Blues_r', top_n))
ax.axvline(len(df_pri)*0.5, color='red',    linestyle='--', alpha=0.7, label='50% coverage')
ax.axvline(len(df_pri)*0.8, color='orange', linestyle='--', alpha=0.7, label='80% coverage')
ax.set_xlabel('등장 연구 수')
ax.set_title(f'Patient Feature 등장 빈도 (상위 {top_n})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../data/eda_feature_coverage.png', dpi=150)
plt.show()

## 6. 텍스트 필드 길이 분포

In [ ]:
text_fields = ['intervention_treatment', 'intervention_control']
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, field in zip(axes, text_fields):
    lengths = df_pri[field].dropna().str.len()
    lengths.hist(bins=60, ax=ax, color='teal', edgecolor='white')
    ax.set_title(f'{field}\n평균={lengths.mean():.0f}자  중앙값={lengths.median():.0f}자', fontsize=10)
    ax.set_xlabel('문자 수')

plt.tight_layout()
plt.savefig('../data/eda_text_length.png', dpi=150)
plt.show()

## 7. 추적 기간 분포

In [ ]:
def extract_months(tf_str):
    if not isinstance(tf_str, str): return None
    tf = tf_str.lower()
    y = re.search(r'(\d+\.?\d*)\s*year', tf)
    if y: return float(y.group(1)) * 12
    m = re.search(r'(\d+\.?\d*)\s*month', tf)
    if m: return float(m.group(1))
    w = re.search(r'(\d+\.?\d*)\s*week', tf)
    if w: return float(w.group(1)) / 4.3
    return None

df_pri['follow_up_months'] = df_pri['time_frame'].apply(extract_months)

fig, ax = plt.subplots(figsize=(10, 4))
df_pri['follow_up_months'].dropna().clip(0, 120).hist(bins=60, ax=ax, color='salmon', edgecolor='white')
ax.set_xlabel('추적 기간 (개월)')
ax.set_title(f'추적 기간 분포  (파싱 성공: {df_pri["follow_up_months"].notna().sum():,}건)', fontsize=12)
ax.xaxis.set_major_locator(ticker.MultipleLocator(12))
plt.tight_layout()
plt.savefig('../data/eda_followup.png', dpi=150)
plt.show()

print(df_pri['follow_up_months'].describe())

## 8. 최종 데이터셋 저장

In [ ]:
# model_df에 원본 PICO 정보 merge
pico_cols = ['pmid','patient_treatment','patient_control','patient_exclusion',
             'intervention_treatment','intervention_control','time_frame']

df_pri['pmid'] = df_pri['pmid'].astype(str)
model_df['pmid'] = model_df['pmid'].astype(str)

# primary source는 pri PICO, secondary도 같은 pmid의 pri PICO 사용
model_final = model_df.merge(df_pri[pico_cols], on='pmid', how='left')

print(f'최종 데이터셋: {len(model_final):,}행 × {model_final.shape[1]}열')
print(model_final.dtypes)

model_final.to_pickle('../data/model_ready_df.pkl')
print('\n→ ../data/model_ready_df.pkl 저장 완료')

In [ ]:
# 최종 요약 출력
print('='*45)
print('       최종 EDA 요약')
print('='*45)
print(f'전체 연구 수           : {len(df_pri):,}')
print(f'Primary 레이블 (파싱)  : {len(pri_results):,}')
print(f'Secondary 레이블 (파싱): {len(sec_results):,}')
print(f'HR/RR/OR + CI + p 보유 : {len(model_final):,}')
print(f'  - Primary            : {(model_final["source"]=="primary").sum():,}')
print(f'  - Secondary          : {(model_final["source"]=="secondary").sum():,}')
print(f'유의한 결과 (p<0.05)   : {model_final["significant"].sum():,} ({model_final["significant"].mean()*100:.1f}%)')
print(f'고유 patient feature   : {len(feature_counter):,}')
print('='*45)